In [1]:
from XGBoostModel import XGBoostModel
from RunData import RunPipeline
import pandas as pd
import optuna
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold ,train_test_split, cross_val_score
import kagglehub
import os
import zipfile
from seed_utils import SEED
import requests


/home/daniel7/.conda/envs/my_env_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# GenericDataPipeline

In [ ]:
class GenericDataPipeline:

    def rank_features(self, X, y, n_folds=5):
        print(f"Starting feature importance calculation with XGBoost and {n_folds}-fold CV...")

        X_copy = X.copy()

        null_ratios = {}
        for col in X_copy.columns:
            null_ratios[col] = X_copy[col].isna().mean()

        for col in X_copy.select_dtypes(include=['int64', 'float64']).columns:
            X_copy[col] = X_copy[col].replace([np.inf, -np.inf], np.nan)
            
            non_null = X_copy[col].dropna()
            if len(non_null) > 0:
                mean_val = non_null.mean()
                std_val = non_null.std()
                if std_val > 0 and not np.isnan(mean_val):
                    upper_bound = mean_val + 5 * std_val
                    lower_bound = mean_val - 5 * std_val
                    X_copy[col] = X_copy[col].clip(lower=lower_bound, upper=upper_bound)
        
        num_cols = X_copy.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = X_copy.select_dtypes(include=['object']).columns
        
        X_processed = X_copy.copy()
        
        for col in cat_cols:
            if X_processed[col].isna().sum() > 0:
                most_freq = X_processed[col].mode()[0] if not X_processed[col].isna().all() else 'Unknown'
                X_processed[col] = X_processed[col].fillna(most_freq)
            X_processed[col] = X_processed[col].astype('category')
        
        for col in num_cols:
            if X_processed[col].isna().sum() > 0:
                median_val = X_processed[col].median() if not X_processed[col].isna().all() else 0
                X_processed[col] = X_processed[col].fillna(median_val)
        
        X_filled = X_processed

        mean_target = float(np.mean(y))
        print(f"Mean target value (for base_score): {mean_target:.6f}")

        base_score = 0.5
        if 0 < mean_target < 1:
            base_score = mean_target

        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'max_depth': 6,
            'eta': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 3,
            'alpha': 0.01,
            'lambda': 1.0,
            'gamma': 0.1,
            'base_score': base_score, 
            'seed': 42,
            'tree_method': 'hist',
            'device': 'cuda'

        }

        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

        fold_importances = []

        fold_scores = []

        print(f"Training XGBoost models across {n_folds} folds...")

        y_values = set(y.unique())
        print(f"Unique target values: {y_values}")

        if not all(isinstance(val, (int, float)) for val in y_values):
            print("Warning: Target variable contains non-numeric values. Converting to numeric.")
            y = y.astype(float)

        if not all(val in [0, 1] for val in y_values):
            print("Warning: Target variable contains values other than 0 and 1.")
            print("Converting target to binary: 0 for negative class, 1 for positive class.")
            y = (y > 0).astype(int)

        # Perform cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_filled, y)):
            print(f"Fold {fold+1}/{n_folds}")

            X_train, X_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            train_pos = np.mean(y_train)
            val_pos = np.mean(y_val)
            print(f"  Train positive rate: {train_pos:.4f}, Validation positive rate: {val_pos:.4f}")

            try:
                dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
                dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

                # Train model
                model = xgb.train(
                    params,
                    dtrain,
                    num_boost_round=100,
                    early_stopping_rounds=20,
                    evals=[(dval, 'validation')],
                    verbose_eval=False
                )

                y_pred = model.predict(dval)
                score = roc_auc_score(y_val, y_pred)
                fold_scores.append(score)

                try:
                    importance = model.get_score(importance_type='gain')

                    fold_importance = {col: importance.get(col, 0) for col in X_filled.columns}

                    max_value = max(fold_importance.values()) if max(fold_importance.values()) > 0 else 1
                    normalized_importance = {col: val/max_value for col, val in fold_importance.items()}

                    fold_importances.append(normalized_importance)
                except Exception as e:
                    print(f"Warning: Could not get feature importance for fold {fold+1}: {str(e)}")

            except Exception as e:
                print(f"Error in fold {fold+1}: {str(e)}")
                print("Skipping this fold and continuing with others.")

        if fold_scores:
            mean_auc = np.mean(fold_scores)
            print(f"Mean AUC across {len(fold_scores)} successful folds: {mean_auc:.4f}")
        else:
            print("No successful folds. Cannot calculate mean AUC.")

        if not fold_importances:
            print("No feature importance information could be gathered from any fold.")
            print("Using a simple fallback method for feature scoring.")

            avg_importance = {}
            for col in X_filled.columns:
                try:
                    valid_mask = ~pd.isna(X_copy[col]) & ~pd.isna(y)
                    if valid_mask.sum() > 10: 
                        if X_filled[col].dtype.name == 'category':
                            col_values = X_filled[col][valid_mask].cat.codes
                        else:
                            col_values = X_filled[col][valid_mask]
                        
                        y_values = y[valid_mask]
                        corr = abs(np.corrcoef(col_values, y_values)[0, 1])
                        avg_importance[col] = corr if not np.isnan(corr) else 0
                    else:
                        avg_importance[col] = np.random.uniform(0, 0.001)
                except Exception as e:
                    print(f"Warning: Could not calculate correlation for {col}: {str(e)}")
                    avg_importance[col] = np.random.uniform(0, 0.001)
        else:
            avg_importance = {}
            for col in X_filled.columns:
                importances = [fold_imp.get(col, 0) for fold_imp in fold_importances]
                if importances:
                    avg_importance[col] = np.mean(importances)
                else:
                    null_ratio = null_ratios.get(col, 0)
                    avg_importance[col] = max(0.001 * (1 - null_ratio), 0.0001)  

        feature_data = []
        for col in X_copy.columns:
            gain_score = avg_importance.get(col, 0)
            null_ratio = null_ratios.get(col, 0)
            feature_data.append({
                'feature_name': col,
                'gain_score': gain_score,
                'null_ratio': null_ratio
            })
        
        feature_df = pd.DataFrame(feature_data)
        feature_df = feature_df.sort_values('gain_score', ascending=False)
        
        print("\nFeature scores:")
        print("-" * 100)
        print(f"{'Rank':<5} | {'Feature':<30} | {'Gain Score':<15} | {'Null Ratio':<10}")
        print("-" * 100)

        for i, row in feature_df.iterrows():
            rank = i + 1
            feat = row['feature_name']
            gain = row['gain_score']
            null_ratio = row['null_ratio']
            print(f"{rank:<5} | {feat:<30} | {gain:.6f} | {null_ratio:.4f}")

        return feature_df

    def objective(self, trial, feature_scores_with_nulls, all_features_score, df, label="readmitted"):
        K = 10
        selected_features = feature_scores_with_nulls['feature_name'].to_list()[:K]
        print("selected_features")
        print(selected_features)
        all_features = all_features_score['feature_name'].to_list()
        # Create binary assignment for each of the selected features:
        assignment = {}
        for feat in selected_features:
            # 1 means feature goes to stage1, 0 means feature is used in stage2
            assignment[feat] = trial.suggest_categorical(f"assign_{feat}", [1, 0])
        
        for feat in all_features:
            if feat not in assignment.keys():
                assignment[feat] = 1
        vals = 0
        for key,value in assignment.items():
            vals +=value
        if vals==0:
            return 99999
        penalty = 0 
        # Derive feature sets based on the assignment:
        stage1_features = [feat for feat, assign in assignment.items() if assign == 1]
        stage2_features = [feat for feat, assign in assignment.items() if assign == 0]
        csv_name = f"trial_{trial.number}_results.csv"
        dm = RunPipeline()
        validation_loss = dm.full_run(df,stage1_features,stage2_features,label,csv_name)
        if validation_loss == 999:
            return validation_loss
        else:
            final_objective = validation_loss + penalty
            return final_objective
    def preprocessing(self,df):
        for col in df.columns:
            if df[col].dtype == 'object':
                # Replace ? with NaN
                df[col] = df[col].replace(['?', ''], np.nan)
                # Convert to categorical codes
                if df[col].isna().sum() < len(df):  # If not all values are NaN
                    df[col] = pd.Categorical(df[col]).codes

                    # Ensure -1 (missing) values are converted to NaN
                    df[col] = df[col].replace(-1, np.nan)
        # Convert boolean columns to int
        for col in df.select_dtypes(include=['bool']).columns:
            df[col] = df[col].astype(int)
        return df 


    def WiDS(self,file_path,n_trials=20):
        print(f"Loading data from {file_path}...")
        df = pd.read_csv(file_path)        
        print("Preprocessing data...")
        column_drop = ['encounter_id', 'patient_id', 'hospital_id','apache_4a_icu_death_prob', 'apache_4a_hospital_death_prob']
        df = df.drop(columns=column_drop)
        df = self.preprocessing(df)
        return df


# WiDS Datathon 2020https://www.kaggle.com/c/widsdatathon2020/data?select=training_v2.csv



In [4]:
file_path = "/sise/home/daniel7/IncrementalLearing/Changes /training_v2.csv"
fs = GenericDataPipeline()
study = fs.WiDS(file_path,20)


Loading data from /sise/home/daniel7/IncrementalLearing/Changes /training_v2.csv...
Preprocessing data...
Using 'hospital_death' as target variable
Starting feature importance calculation with XGBoost and 5-fold CV...
Mean target value (for base_score): 0.086302
Training XGBoost models across 5 folds...
Unique target values: {np.int64(0), np.int64(1)}
Fold 1/5
  Train positive rate: 0.0863, Validation positive rate: 0.0863
Fold 2/5
  Train positive rate: 0.0863, Validation positive rate: 0.0863
Fold 3/5
  Train positive rate: 0.0863, Validation positive rate: 0.0863
Fold 4/5
  Train positive rate: 0.0863, Validation positive rate: 0.0863
Fold 5/5
  Train positive rate: 0.0863, Validation positive rate: 0.0863


[I 2025-05-18 09:15:25,617] A new study created in memory with name: no-name-59225d53-b5c0-47d3-91d0-3238f30d5602


Mean AUC across 5 successful folds: 0.9015

Feature scores:
----------------------------------------------------------------------------------------------------
Rank  | Feature                        | Gain Score      | Null Ratio
----------------------------------------------------------------------------------------------------
41    | ventilated_apache              | 1.000000 | 0.0078
116   | d1_lactate_min                 | 0.514493 | 0.7458
25    | gcs_motor_apache               | 0.430733 | 0.0207
24    | gcs_eyes_apache                | 0.307329 | 0.0207
3     | elective_surgery               | 0.277335 | 0.0000
115   | d1_lactate_max                 | 0.255592 | 0.7458
64    | d1_sysbp_min                   | 0.183266 | 0.0017
27    | gcs_verbal_apache              | 0.158671 | 0.0207
100   | d1_bun_min                     | 0.146323 | 0.1146
17    | apache_3j_diagnosis            | 0.143022 | 0.0120
66    | d1_sysbp_noninvasive_min       | 0.134128 | 0.0112
99    | d1_bun_max 

[I 2025-05-18 09:15:25,970] Trial 0 finished with value: 999.0 and parameters: {'assign_d1_lactate_min': 1, 'assign_d1_lactate_max': 0, 'assign_d1_bun_min': 0, 'assign_d1_bun_max': 0, 'assign_d1_arterial_ph_min': 1, 'assign_d1_pao2fio2ratio_min': 0, 'assign_fio2_apache': 1, 'assign_d1_pao2fio2ratio_max': 0, 'assign_d1_albumin_min': 0, 'assign_d1_arterial_pco2_min': 0}. Best is trial 0 with value: 999.0.


Train set distribution:
hospital_death  has_extended
0               0                5975
                1               61063
1               0                 375
                1                5957
dtype: int64

Test set distribution:
hospital_death  has_extended
0               0                1494
                1               15266
1               0                  94
                1                1489
dtype: int64

⚠️ One of the train/test groups has fewer than 100 samples!
selected_features
['d1_lactate_min', 'd1_lactate_max', 'd1_bun_min', 'd1_bun_max', 'd1_arterial_ph_min', 'd1_pao2fio2ratio_min', 'fio2_apache', 'd1_pao2fio2ratio_max', 'd1_albumin_min', 'd1_arterial_pco2_min']

=== Breakdown BEFORE splitting ===
has_extended
1    83423
0     8290
Name: count, dtype: int64
Extended percentage: 90.96 %
Train set distribution:
hospital_death  has_extended
0               0                6224
                1               60814
1               0                 408


...

********** run results **********
{'max_depth': 6, 'learning_rate': 0.0014174436866040143, 'n_estimators': 702, 'min_child_weight': 2, 'subsample': 0.7711391599218456, 'colsample_bytree': 0.7460829963609165, 'gamma': 2.4995160218396766, 'lambda': 7.864319505643913, 'alpha': 3.2803023907284805, 'scale_pos_weight': 8.657435951885235, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.8794230809419262), np.float64(0.880350741918115), np.float64(0.8792558989027893)]
********** run results **********


/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
[I 2025-05-18 12:06:25,894] Trial 19 finished with value: 0.8769955961065697 and parameters: {'max_depth': 3, 'learning_rate': 0.0030978013310226245, 'n_estimators': 878, 'min_child_weight': 1, 'subsample': 0.843277135672379, 'colsample_bytree': 0.6690645962826435, 'gamma': 1.548785550327568, 'lambda': 9.174377539724937, 'alpha': 1.1002421168304044, 'scale_pos_weight': 4.849475044614819}. Best is trial 10 with value: 0.852779163483135.


********** run results **********
{'max_depth': 3, 'learning_rate': 0.0030978013310226245, 'n_estimators': 878, 'min_child_weight': 1, 'subsample': 0.843277135672379, 'colsample_bytree': 0.6690645962826435, 'gamma': 1.548785550327568, 'lambda': 9.174377539724937, 'alpha': 1.1002421168304044, 'scale_pos_weight': 4.849475044614819, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.8747693145447247), np.float64(0.879291751269271), np.float64(0.8769257225057135)]
********** run results **********
Best parameters found: {'max_depth': 3, 'learning_rate': 0.0010177896885449403, 'n_estimators': 794, 'min_child_weight': 3, 'subsample': 0.7303572379547679, 'colsample_bytree': 0.6056568774110992, 'gamma': 2.0830367963392997, 'lambda': 9.986943546690776, 'alpha': 0.5102138101939706, 'scale_pos_weight': 9.92418369929199}
Best CV AUC score: 0.8528

===== Detailed Cross-Validation Results with Best Parameters =====
Params: {'max_depth': 3, 'learnin

[I 2025-05-18 12:11:42,715] Trial 19 finished with value: 0.06805017010011316 and parameters: {'assign_d1_lactate_min': 1, 'assign_d1_lactate_max': 1, 'assign_d1_bun_min': 1, 'assign_d1_bun_max': 0, 'assign_d1_arterial_ph_min': 0, 'assign_d1_pao2fio2ratio_min': 1, 'assign_fio2_apache': 1, 'assign_d1_pao2fio2ratio_max': 1, 'assign_d1_albumin_min': 0, 'assign_d1_arterial_pco2_min': 0}. Best is trial 1 with value: -0.17931811150483568.



Extended model (with extended)
AUC: 0.8149, Accuracy: 0.9112, F1 Score: 0.0000

Combined model (no extended)
AUC: 0.8880, Accuracy: 0.9264, F1 Score: 0.5120

Combined model (with extended)
AUC: 0.8488, Accuracy: 0.7871, F1 Score: 0.3835

Results Summary:
                            Model       AUC  Accuracy        F1  \
0        Base model (no extended)  0.853814  0.938480  0.000000   
1  Extended model (with extended)  0.814943  0.911238  0.000000   
2    Combined model (no extended)  0.887992  0.926417  0.512000   
3  Combined model (with extended)  0.848815  0.787054  0.383481   

                                                                                                                                                                                                                                                                                                                                                                                                                        

# Final Results

AUC Summary:
                         Model      AUC
      Base model (no extended) 0.896479
Extended model (with extended) 0.845295
  Combined model (no extended) 0.806247
Combined model (with extended) 0.756210

Base Features:
d1_lactate_min, d1_lactate_max, d1_bun_min, ..., lymphoma

Extended Features:
d1_bun_max, d1_arterial_ph_min, d1_pao2fio2ratio_max, d1_albumin_min, d1_arterial_pco2_min, has_extended

Target: hospital_death

AUC Differences:
Extended model - Combined (with extended): 0.08908499999999997
Base model - Combined (no extended):       0.09023199999999998

Standard Deviations:
Extended model - Std Dev: 0.0051
Base model - Std Dev:     0.0052
